In [6]:
# ! pip install -U transformers datasets accelerate scikit-learn pyarrow -U
# !pip install transformers[torch]

Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 42.9 MB/s  0:00:0040.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 94.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 50.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 39.5 MB/s  0:00:019.1 MB/s eta 0:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 42.6 MB/s  0:00:00
  Attempting uninstall: hf-xet8;5;237m╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/18 [pyarrow]
    Found existing installation: hf-xet 1.1.2;237m━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/18 [pyarrow]
    Uninstalling hf-xet-1.1.2:237m╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/18 [pyarrow]
      Successfully uninstalled hf-xet-1.1.2;5;237m━━━━━━━━━━━━━━━━━━━━━

# Imports

In [1]:
import os
os.environ["CUDA_DEVICE_ORDER"]    = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"        # the largest/fastest GPU
os.environ["TORCHDYNAMO_DISABLE"]  = "1"        # disable torch.compile/inductor

In [2]:
import json
import random
import torch
import numpy as np
from functools import partial
from collections import defaultdict
from typing import Dict, List, Tuple, Any

from datasets import Dataset
from torch.nn import CrossEntropyLoss
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    DataCollatorForTokenClassification, Trainer, TrainingArguments, EarlyStoppingCallback
)

In [3]:
parsed = []

with open("../dataset/rulesm_synth.json") as f:
    data = json.load(f)

for d in data:
    try:
        parsed.append(eval(d.replace('```json', '').replace('```', '')))
    except Exception as E:
        print(E)

random.shuffle(parsed)

unterminated string literal (detected at line 11) (<string>, line 11)


## Functions

In [4]:
def merge_labeled_spans(spans, merge_labels=("contradiction", "paraphrase")):
    """
    spans: список словарей вида {'start': int, 'end': int, 'label': str}
    Объединяет подряд идущие touching/overlapping интервалы одного и того же лейбла
    из merge_labels. Порядок берётся как в исходном списке.
    """
    if not spans:
        return []

    out = []
    buf = None  # {'start':..., 'end':..., 'label':...}

    def flush():
        nonlocal buf
        if buf is not None:
            out.append(buf)
            buf = None

    for s in spans:
        lab = s["label"]
        st, en = int(s["start"]), int(s["end"])

        if lab in merge_labels:
            if buf is None:
                buf = {"start": st, "end": en, "label": lab}
            elif buf["label"] == lab and st <= buf["end"] + 1:
                # склеиваем соприкасающиеся/перекрывающиеся интервалы
                if st < buf["start"]:
                    buf["start"] = st
                if en > buf["end"]:
                    buf["end"] = en
            else:
                flush()
                buf = {"start": st, "end": en, "label": lab}
        else:
            flush()
            out.append(s)

    flush()
    return out

In [5]:
def build_text_and_spans(items: List[Dict], side: str) -> Tuple[str, List[Dict]]:
    """
    side: 'input' или 'new_input'
    Возвращает: (text, spans), где spans — список {'start': int, 'end': int, 'label': str}
    """
    assert side in ("input", "new_input")
    parts = [it.get(side, "") for it in items]
    text = " ".join(parts)

    # какие метки учитывать на каждой стороне
    if side == "input":
        keep = {"paraphrase", "contradiction"}
    else:  # new_input
        keep = {"paraphrase", "contradiction", "addition"}

    spans = []
    offset = 0
    for i, it in enumerate(items):
        seg = it.get(side, "")
        seg_len = len(seg)
        start = offset
        end = start + seg_len

        lbl = it.get("label", "")
        if seg_len > 0 and lbl in keep:
            spans.append({"start": start, "end": end, "label": lbl})

        # сдвиг на сегмент + пробел после него (кроме последнего)
        offset = end
        if i < len(items) - 1:
            offset += 1  # пробел между частями

    return text, merge_labeled_spans(spans)


def make_two_samples(items: List[Dict]) -> Dict[str, Dict]:
    """
    Из одного исходного примера делает два семпла:
      - sample_input: текст = конкатенация всех 'input', спаны по правилам выше
      - sample_new_input: текст = конкатенация всех 'new_input', спаны по правилам выше
    """
    text_input, spans_input = build_text_and_spans(items, side="input")
    text_new, spans_new = build_text_and_spans(items, side="new_input")

    return ({
        'context': text_input,
        'answer': text_new,
       'labels': spans_new,
    },{
            'context': text_new,
            'answer': text_input,
           'labels': spans_input
            })

samples = []
for p in parsed:
    samples.extend(make_two_samples(p))

In [6]:
samples[10]

{'context': 'Командиры воинских частей, из которых прибыли осужденные военнослужащие, обязаны поддерживать постоянную связь с командиром дисциплинарной воинской части, интересоваться поведением бывших подчиненных и оказывать содействие в их исправлении.',
 'answer': 'Командиры частей, откуда поступили военные преступники, могут поддерживать нерегулярную связь с командиром дисциплинарной части, узнавать о поведении своих бывших подчиненных и предоставлять помощь в их исправлении.',
 'labels': [{'start': 0, 'end': 55, 'label': 'paraphrase'},
  {'start': 56, 'end': 128, 'label': 'contradiction'},
  {'start': 129, 'end': 215, 'label': 'paraphrase'}]}

# Train

In [22]:
# ====== разрез на train/test (последние 10% в тест) ======
def split_train_test(samples):
    n = len(samples)
    n_test = max(1, int(round(0.10 * n))) if n > 1 else n
    train = samples[:-n_test] if n_test < n else []
    test  = samples[-n_test:]
    return train, test

def encode_example(ex, tokenizer, max_length):
    """
    Токенизируем как пару (context, answer), и размечаем ЛИШЬ токены второй последовательности (answer).
    Лейбл токена — тот класс спана, с которым наибольшее перекрытие по символам; если перекрытий нет — 'O'.
    Контекст и спецтокены -> -100 (игнор в лоссе).
    """
    enc = tokenizer(
        ex["context"], ex["answer"],
        truncation=True, 
        max_length=max_length,
        return_offsets_mapping=True
    )

    seq_ids = enc.sequence_ids()
    offsets = enc["offset_mapping"]
    spans = sorted(ex.get("labels", []), key=lambda s: (s["start"], s["end"]))
    lab_ids = []

    for sid, (st, en) in zip(seq_ids, offsets):
        # игнор спецтокенов (None) и токенов контекста (sid==0)
        if sid != 1 or (st == en):
            lab_ids.append(-100)
            continue

        # находим лучший спан по длине пересечения
        best_lbl = "O"
        best_ov = 0
        for sp in spans:
            s, e = int(sp["start"]), int(sp["end"])
            ov = max(0, min(en, e) - max(st, s))
            if ov > best_ov:
                best_ov = ov
                best_lbl = sp["label"]

        lab_ids.append(label2id.get(best_lbl, label2id["O"]))

    enc.pop("offset_mapping")
    enc["labels"] = lab_ids
    return enc

def build_hf_dataset(raw_list, tokenizer, max_length):
    """Создаем датасет с помощью любого токенизатора"""
    ds = Dataset.from_list(raw_list)
    tokenize = partial(encode_example, tokenizer=tokenizer, max_length=max_length)
    return ds.map(tokenize, remove_columns=ds.column_names)

In [23]:
# ====== метрики ======
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    y_true, y_pred = [], []
    for p_row, l_row in zip(preds, labels):
        for p, l in zip(p_row, l_row):
            if l == -100:  # игнорим контекст/спецтокены
                continue
            y_true.append(l)
            y_pred.append(p)

    # macro по трём классам (без O)
    target_ids = [label2id["paraphrase"], label2id["contradiction"], label2id["addition"]]
    # фильтруем только метки НЕ 'O'
    mask = [t in target_ids for t in y_true]
    yt = [t for t, m in zip(y_true, mask) if m]
    yp = [p for p, m in zip(y_pred, mask) if m]

    if len(yt) == 0:
        f1_macro = prec_macro = rec_macro = 0.0
    else:
        prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
            yt, yp, labels=target_ids, average="macro", zero_division=0
        )

    acc_all = accuracy_score(y_true, y_pred) if len(y_true) else 0.0

    return {
        "f1_macro": f1_macro,
        "precision_macro": prec_macro,
        "recall_macro": rec_macro,
        "acc_all_tokens": acc_all,
    }

## Cross Entropy Loss

In [43]:
def compute_class_weights(train_ds, num_labels: int, ignore_index: int = -100):
    counts = np.zeros(num_labels, dtype=np.int64)

    for ex in train_ds:
        labels = np.array(ex["labels"], dtype=np.int64)
        labels = labels[labels != ignore_index]
        if labels.size == 0:
            continue
        uniq, freq = np.unique(labels, return_counts=True)
        counts[uniq] += freq

    # чтобы не делить на 0
    counts = np.maximum(counts, 1)

    # частоты
    freqs = counts / counts.sum()

    # инверсные веса (реже класс → выше вес)
    inv = 1.0 / freqs
    weights = inv / inv.mean()  # нормализуем, чтобы средний вес ≈ 1.0

    return weights.astype(np.float32).tolist()

In [44]:
class WeightedTokenCETrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights is not None:
            # сохраняем в тензоре, device подгоним в compute_loss
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32)
        else:
            self.class_weights = None

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs: bool = False,
        **kwargs,  # ловим num_items_in_batch и прочее
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits  # [B, T, C]

        if self.class_weights is not None:
            weight = self.class_weights.to(logits.device)
        else:
            weight = None

        loss_fct = CrossEntropyLoss(
            weight=weight,
            ignore_index=-100,  # как и раньше
        )

        # CrossEntropy ждёт [N, C] и [N]
        loss = loss_fct(
            logits.view(-1, logits.size(-1)),
            labels.view(-1),
        )

        if return_outputs:
            return loss, outputs
        return loss

## Focal Loss

In [22]:
import torch.nn.functional as F

class FocalLoss(torch.nn.Module):
    def __init__(self, gamma=2.0, alpha=None, ignore_index=-100):
        super().__init__()
        self.gamma = gamma
        self.ignore_index = ignore_index
        if isinstance(alpha, (list, tuple)):
            self.alpha = torch.tensor(alpha)
        else:
            self.alpha = alpha

    def forward(self, logits, targets):
        # logits: [B, T, C], targets: [B, T]
        B, T, C = logits.shape
        logits = logits.view(-1, C)
        targets = targets.view(-1)

        mask = targets != self.ignore_index
        logits = logits[mask]
        targets = targets[mask]
        if targets.numel() == 0:
            return logits.sum() * 0

        log_probs = F.log_softmax(logits, dim=-1)  # [N, C]
        probs = log_probs.exp()
        targets_one_hot = F.one_hot(targets, num_classes=C)

        pt = (targets_one_hot * probs).sum(dim=-1)  # [N]
        log_pt = (targets_one_hot * log_probs).sum(dim=-1)

        if self.alpha is not None:
            alpha_t = self.alpha.to(logits.device)[targets]
        else:
            alpha_t = 1.0

        loss = -alpha_t * (1 - pt) ** self.gamma * log_pt
        return loss.mean()

from transformers import Trainer

class FocalTokenTrainer(Trainer):
    def __init__(self, *args, focal_gamma=2.0, focal_alpha=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.focal_loss = FocalLoss(gamma=focal_gamma, alpha=focal_alpha, ignore_index=-100)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # labels вынимаем из inputs, чтобы модель не пыталась сама считать loss
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits  # [B, T, C]
        loss = self.focal_loss(logits, labels)
        
        if return_outputs:
            return loss, outputs
        return loss

## Train

In [8]:
def train_token_classifier(samples, model_name, use_weighted_ce=False, use_focal=False):

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    max_len = min(MAX_LENGTH, 512)
    
    train_raw, test_raw = split_train_test(samples)
    train_ds = build_hf_dataset(train_raw, tokenizer=tokenizer, max_length=max_len)
    test_ds  = build_hf_dataset(test_raw, tokenizer=tokenizer, max_length=max_len)

    model = AutoModelForTokenClassification.from_pretrained(
        model_name, num_labels=len(LABELS), id2label=id2label, label2id=label2id
    )

    args = TrainingArguments(
        output_dir="mbert_tokcls_out",
        learning_rate=LR,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        num_train_epochs=EPOCHS,
        eval_strategy="steps",   # <- исправлено с eval_strategy
        save_strategy="steps",
        eval_steps=250,
        save_steps=250,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,            # хранить только 1 чекпоинт
        weight_decay=0.01,
        logging_steps=50,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
    )

    data_collator = DataCollatorForTokenClassification(tokenizer)

    if use_focal:
        trainer = FocalTokenTrainer(
            model=model,
            args=args,
            train_dataset=train_ds if len(train_ds) else None,
            eval_dataset=test_ds if len(test_ds) else None,
            tokenizer=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=6)],
            focal_gamma=2.0,
            focal_alpha=[0.2, 1.0, 1.0, 1.0],  # например, меньше вес для 'O'
        )
    elif use_weighted_ce:
        class_weights = compute_class_weights(train_ds, num_labels=len(LABELS))
        print("Class weights:", dict(zip(LABELS, class_weights)))

        trainer = WeightedTokenCETrainer(
            model=model,
            args=args,
            train_dataset=train_ds if len(train_ds) else None,
            eval_dataset=test_ds if len(test_ds) else None,
            processing_class=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=6)],
            class_weights=class_weights,
        )
    else:
        trainer = Trainer(
            model=model,
            args=args,
            train_dataset=train_ds if len(train_ds) else None,
            eval_dataset=test_ds if len(test_ds) else None,
            tokenizer=tokenizer,
            data_collator=data_collator,
            compute_metrics=compute_metrics,  # считает macro-F1/precision/recall и acc
            callbacks=[EarlyStoppingCallback(early_stopping_patience=6)],
        )
    print(f"using {trainer.__class__.__name__}")

    if len(train_ds):
        trainer.train()

    print("\n== Eval on held-out 10% ==")
    metrics = trainer.evaluate(test_ds) if len(test_ds) else {}
    for k, v in metrics.items():
        if k.startswith("eval_"):
            print(f"{k}: {v:.4f}")

    # ---- дополнительно: F1 по каждому классу (paraphrase/contradiction/addition) ----
    f1_by_class = {}
    if len(test_ds):
        pred = trainer.predict(test_ds)
        logits = pred.predictions
        labels = pred.label_ids
        preds = np.argmax(logits, axis=-1)

        y_true, y_pred = [], []
        for p_row, l_row in zip(preds, labels):
            for p, l in zip(p_row, l_row):
                if l == -100:     # игнорируем спецтокены/контекст
                    continue
                y_true.append(int(l))
                y_pred.append(int(p))

        target_ids = [label2id["paraphrase"], label2id["contradiction"], label2id["addition"]]
        # считаем метрики только по токенам, принадлежащим целевым классам (без 'O')
        y_true_t = [t for t in y_true if t in target_ids]
        y_pred_t = [p for p, t in zip(y_pred, y_true) if t in target_ids]

        if len(y_true_t):
            prec, rec, f1, support = precision_recall_fscore_support(
                y_true_t, y_pred_t, labels=target_ids, average=None, zero_division=0
            )
            id2label_local = {v: k for k, v in label2id.items()}
            print("\n== Per-class F1 (без 'O') ==")
            for cls_id, f1v, sup in zip(target_ids, f1, support):
                cls_name = id2label_local[cls_id]
                f1_by_class[cls_name] = float(f1v)
                print(f"{cls_name:14s}: F1={f1v:.4f} (support={sup})")
        else:
            print("\nНет токенов целевых классов в тесте — per-class F1 не вычислить.")

    # сохраняем лучший
    trainer.save_model(f"{model_name}-best")
    tokenizer.save_pretrained(f"{model_name}-best")

    # вернём и сводные метрики, и per-class F1
    return trainer, metrics, f1_by_class

## Training

In [20]:
# ====== настройки ======
MAX_LENGTH = 8192
LR = 1e-6
EPOCHS = 15
BATCH = 8
SEED = 42
MODEL_NAME = "intfloat/multilingual-e5-large-instruct" #"ai-forever/ruBert-large" #"ai-forever/ruBert-base" #"jhu-clsp/mmBERT-base"

LABELS = ["O", "paraphrase", "contradiction", "addition"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

### Focal

In [25]:
trainer, test_metrics, f1_by_class = train_token_classifier(samples, use_focal=True)

Map:   0%|          | 0/5816 [00:00<?, ? examples/s]

Map:   0%|          | 0/646 [00:00<?, ? examples/s]

Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at jhu-clsp/mmBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_946171/465849211.py:44: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `FocalTokenTrainer.__init__`. Use `processing_class` instead.
  super().__init__(*args, **kwargs)


using <__main__.FocalTokenTrainer object at 0x7fd2c53826d0>


Step,Training Loss,Validation Loss,F1 Macro,Precision Macro,Recall Macro,Acc All Tokens
250,0.391100,0.365248,0.379679,0.475622,0.377596,0.511411
500,0.261300,0.249797,0.568934,0.661617,0.531527,0.604146
750,0.174400,0.174624,0.726904,0.755149,0.706178,0.712493
1000,0.138200,0.134322,0.807833,0.809355,0.806460,0.789470
1250,0.114100,0.121043,0.831735,0.845571,0.824560,0.814383
1500,0.091400,0.104775,0.857230,0.864725,0.851357,0.838121
1750,0.083100,0.097734,0.873517,0.875361,0.871817,0.855421
2000,0.087800,0.090120,0.885292,0.885910,0.884821,0.867731
2250,0.075300,0.086806,0.893019,0.893826,0.892242,0.875142
2500,0.057800,0.086316,0.897687,0.896978,0.898539,0.880664



== Eval on held-out 10% ==


eval_loss: 0.1229
eval_f1_macro: 0.9220
eval_precision_macro: 0.9225
eval_recall_macro: 0.9216
eval_acc_all_tokens: 0.9048
eval_runtime: 2.3920
eval_samples_per_second: 270.0630
eval_steps_per_second: 33.8620

== Per-class F1 (без 'O') ==
paraphrase    : F1=0.9053 (support=26245)
contradiction : F1=0.8962 (support=24794)
addition      : F1=0.9647 (support=3203)


### CE weighted

In [49]:
trainer, test_metrics, f1_by_class = train_token_classifier(samples, use_weighted_ce=True)

Map:   0%|          | 0/5816 [00:00<?, ? examples/s]

Map:   0%|          | 0/646 [00:00<?, ? examples/s]

Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at jhu-clsp/mmBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Class weights: {'O': 3.6301660537719727, 'paraphrase': 0.041072387248277664, 'contradiction': 0.04085487499833107, 'addition': 0.2879066467285156}
using WeightedTokenCETrainer


Step,Training Loss,Validation Loss,F1 Macro,Precision Macro,Recall Macro,Acc All Tokens
250,0.812900,0.817734,0.414611,0.411029,0.526638,0.449899
500,0.518600,0.510659,0.603017,0.564704,0.686988,0.593799
750,0.359400,0.362368,0.753271,0.730392,0.782522,0.723555
1000,0.292200,0.271903,0.824529,0.803907,0.850419,0.805082
1250,0.224600,0.241635,0.850478,0.838418,0.870755,0.833003
1500,0.184000,0.209911,0.876002,0.863368,0.890198,0.857439
1750,0.175400,0.199134,0.882187,0.869728,0.898013,0.866171
2000,0.184900,0.181921,0.895927,0.886622,0.906719,0.877766
2250,0.164900,0.172333,0.898550,0.885962,0.912649,0.883654
2500,0.120400,0.171414,0.905561,0.898702,0.913508,0.889341



== Eval on held-out 10% ==


eval_loss: 0.1777
eval_f1_macro: 0.9219
eval_precision_macro: 0.9167
eval_recall_macro: 0.9278
eval_acc_all_tokens: 0.9073
eval_runtime: 2.4393
eval_samples_per_second: 264.8270
eval_steps_per_second: 33.2060

== Per-class F1 (без 'O') ==
paraphrase    : F1=0.9091 (support=26245)
contradiction : F1=0.8973 (support=24794)
addition      : F1=0.9594 (support=3203)


### Base

In [12]:
trainer, test_metrics, f1_by_class = train_token_classifier(samples)

Map:   0%|          | 0/5816 [00:00<?, ? examples/s]

Map:   0%|          | 0/646 [00:00<?, ? examples/s]

Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at jhu-clsp/mmBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_919094/2372633084.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,F1 Macro,Precision Macro,Recall Macro,Acc All Tokens
250,0.928900,0.917874,0.362615,0.463637,0.368153,0.504473
500,0.753300,0.722080,0.572123,0.681903,0.538399,0.637856
750,0.485900,0.473800,0.789385,0.792627,0.786268,0.791334
1000,0.368200,0.391561,0.838015,0.836053,0.843710,0.832477
1250,0.321100,0.335341,0.864780,0.855427,0.875124,0.859240
1500,0.284600,0.332815,0.871958,0.872302,0.877122,0.858904
1750,0.247400,0.293189,0.891040,0.889878,0.892942,0.880736
2000,0.212700,0.278952,0.898545,0.900257,0.896979,0.886114
2250,0.180500,0.269864,0.906256,0.909653,0.903257,0.893996
2500,0.147100,0.278569,0.907505,0.910019,0.905316,0.894668



== Eval on held-out 10% ==


eval_loss: 0.3017
eval_f1_macro: 0.9239
eval_precision_macro: 0.9257
eval_recall_macro: 0.9222
eval_acc_all_tokens: 0.9104
eval_runtime: 2.1847
eval_samples_per_second: 295.6860
eval_steps_per_second: 37.0750

== Per-class F1 (без 'O') ==
paraphrase    : F1=0.9068 (support=24263)
contradiction : F1=0.9064 (support=25456)
addition      : F1=0.9586 (support=3537)


In [13]:
f1_by_class

{'paraphrase': 0.906823335394764,
 'contradiction': 0.9063726259613876,
 'addition': 0.9586118617550846}

In [13]:
# ====== настройки ======
MAX_LENGTH = 8192
LR = 1e-6
EPOCHS = 25
BATCH = 8
SEED = 42

LABELS = ["O", "paraphrase", "contradiction", "addition"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [66]:
trainer, test_metrics, f1_by_class = train_token_classifier(samples, use_weighted_ce=False, use_focal=False)

Map:   0%|          | 0/5816 [00:00<?, ? examples/s]

Map:   0%|          | 0/646 [00:00<?, ? examples/s]

Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at jhu-clsp/mmBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_946171/3599954242.py:62: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


using Trainer


Step,Training Loss,Validation Loss,F1 Macro,Precision Macro,Recall Macro,Acc All Tokens
250,0.919700,0.889553,0.377051,0.503141,0.380143,0.524344
500,0.737400,0.718547,0.564424,0.676483,0.533191,0.634948
750,0.467500,0.469728,0.776545,0.798115,0.759010,0.790956
1000,0.367200,0.378324,0.839828,0.840981,0.838815,0.840451
1250,0.314400,0.348921,0.860786,0.864342,0.859498,0.856063
1500,0.266000,0.317093,0.877686,0.877472,0.877935,0.870152
1750,0.236400,0.306705,0.888183,0.888234,0.888385,0.879196
2000,0.255000,0.292793,0.895331,0.894164,0.896662,0.884737
2250,0.218500,0.287355,0.899801,0.899461,0.900229,0.889030
2500,0.166200,0.292887,0.903818,0.900355,0.907386,0.892864



== Eval on held-out 10% ==


eval_loss: 0.3874
eval_f1_macro: 0.9235
eval_precision_macro: 0.9256
eval_recall_macro: 0.9214
eval_acc_all_tokens: 0.9072
eval_runtime: 2.4502
eval_samples_per_second: 263.6540
eval_steps_per_second: 33.0590

== Per-class F1 (без 'O') ==
paraphrase    : F1=0.9070 (support=26245)
contradiction : F1=0.8994 (support=24794)
addition      : F1=0.9640 (support=3203)


In [ ]:
trainer, test_metrics, f1_by_class = train_token_classifier(samples)

Map:   0%|          | 0/6106 [00:00<?, ? examples/s]

Map:   0%|          | 0/678 [00:00<?, ? examples/s]

Some weights of ModernBertForTokenClassification were not initialized from the model checkpoint at jhu-clsp/mmBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_789/1656772971.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,F1 Macro,Precision Macro,Recall Macro,Acc All Tokens
250,0.954000,0.962495,0.346237,0.431080,0.355240,0.497792
500,0.856600,0.829466,0.427092,0.567122,0.420034,0.565925
750,0.686300,0.663490,0.633406,0.685828,0.604416,0.682096
1000,0.466900,0.489431,0.763981,0.799812,0.739992,0.783489
1250,0.419200,0.402516,0.824462,0.824917,0.824157,0.830355
1500,0.330200,0.366423,0.845081,0.856570,0.835350,0.848191
1750,0.332500,0.336539,0.859753,0.875415,0.846164,0.861831
2000,0.296200,0.312885,0.873775,0.879707,0.868322,0.872720
2250,0.261300,0.309747,0.880835,0.886068,0.877036,0.875539
2500,0.274700,0.289679,0.887562,0.898374,0.877948,0.883880


In [21]:
MODEL_NAME

'ai-forever/ruBert-base'

In [25]:
trainer, test_metrics, f1_by_class = train_token_classifier(samples, model_name=MODEL_NAME, use_weighted_ce=False, use_focal=False)

Map:   0%|          | 0/5816 [00:00<?, ? examples/s]

Map:   0%|          | 0/646 [00:00<?, ? examples/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at ai-forever/ruBert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_1175400/1835346597.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


using Trainer


Step,Training Loss,Validation Loss,F1 Macro,Precision Macro,Recall Macro,Acc All Tokens
250,0.892400,0.896306,0.319663,0.308542,0.332442,0.462810
500,0.841900,0.823828,0.354904,0.626775,0.361334,0.486651
750,0.761700,0.752073,0.592175,0.598982,0.602280,0.561663
1000,0.727600,0.716039,0.639810,0.636097,0.650171,0.593037
1250,0.711200,0.695602,0.637932,0.647338,0.677128,0.600192
1500,0.672600,0.665479,0.685314,0.678429,0.696780,0.636320
1750,0.652900,0.636486,0.707326,0.695981,0.721495,0.663245
2000,0.641600,0.625055,0.715138,0.706582,0.730183,0.671361
2250,0.613500,0.608911,0.731255,0.720884,0.745412,0.689387
2500,0.589900,0.605958,0.733641,0.735764,0.743143,0.692522



== Eval on held-out 10% ==


eval_loss: 0.4928
eval_f1_macro: 0.8120
eval_precision_macro: 0.8126
eval_recall_macro: 0.8153
eval_acc_all_tokens: 0.7821
eval_runtime: 1.2482
eval_samples_per_second: 517.5350
eval_steps_per_second: 64.8920

== Per-class F1 (без 'O') ==
paraphrase    : F1=0.7636 (support=18255)
contradiction : F1=0.7826 (support=18470)
addition      : F1=0.8898 (support=2829)


In [14]:
MODEL_NAME

'ai-forever/ruBert-large'

In [ ]:
trainer, test_metrics, f1_by_class = train_token_classifier(samples, model_name=MODEL_NAME, use_weighted_ce=False, use_focal=False)

Map:   0%|          | 0/5816 [00:00<?, ? examples/s]

Map:   0%|          | 0/646 [00:00<?, ? examples/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at ai-forever/ruBert-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_3485373/740746523.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


using Trainer


Step,Training Loss,Validation Loss,F1 Macro,Precision Macro,Recall Macro,Acc All Tokens
250,0.820500,0.810678,0.382570,0.647908,0.385925,0.522475
500,0.739900,0.723029,0.627692,0.624179,0.632796,0.600431
750,0.700500,0.675910,0.667690,0.663250,0.677780,0.643668
1000,0.644900,0.640395,0.699294,0.692374,0.706861,0.674362
1250,0.616100,0.623427,0.713225,0.705729,0.727690,0.686956
1500,0.553000,0.608500,0.735944,0.734273,0.738302,0.708899
1750,0.532100,0.583514,0.747792,0.738036,0.758627,0.725597
2000,0.503700,0.576558,0.760784,0.756884,0.766344,0.738503
2250,0.475500,0.580204,0.766946,0.771169,0.773513,0.748345
2500,0.451500,0.544102,0.786729,0.794922,0.785116,0.766107


In [24]:
MODEL_NAME

'intfloat/multilingual-e5-large-instruct'

In [25]:
trainer, test_metrics, f1_by_class = train_token_classifier(samples, model_name=MODEL_NAME, use_weighted_ce=False, use_focal=False)

Map:   0%|          | 0/5816 [00:00<?, ? examples/s]

Map:   0%|          | 0/646 [00:00<?, ? examples/s]

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at intfloat/multilingual-e5-large-instruct and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_3505995/740746523.py:66: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


using Trainer


Step,Training Loss,Validation Loss,F1 Macro,Precision Macro,Recall Macro,Acc All Tokens
250,0.923900,0.909679,0.316072,0.312781,0.336056,0.465489
500,0.886900,0.885372,0.335043,0.326237,0.350563,0.486778
750,0.853500,0.853696,0.345139,0.344609,0.366822,0.508107
1000,0.816900,0.819486,0.372321,0.359343,0.387048,0.539833
1250,0.799100,0.805960,0.365021,0.386805,0.396334,0.547785
1500,0.790000,0.767477,0.476381,0.644234,0.461852,0.573588
1750,0.756400,0.743400,0.591003,0.627689,0.576900,0.599225
2000,0.748400,0.731423,0.612214,0.658805,0.596448,0.604403
2250,0.711500,0.703211,0.637131,0.654306,0.630897,0.621653
2500,0.710400,0.713900,0.627375,0.693548,0.628109,0.611506



== Eval on held-out 10% ==


eval_loss: 0.4483
eval_f1_macro: 0.8202
eval_precision_macro: 0.8261
eval_recall_macro: 0.8151
eval_acc_all_tokens: 0.8121
eval_runtime: 2.7842
eval_samples_per_second: 232.0270
eval_steps_per_second: 29.0930

== Per-class F1 (без 'O') ==
paraphrase    : F1=0.8148 (support=22840)
contradiction : F1=0.8052 (support=21992)
addition      : F1=0.8405 (support=3457)


# inference

In [26]:
import torch
from transformers import AutoTokenizer, AutoModelForTokenClassification

# === загрузка лучшего чекпоинта один раз ===
CKPT_DIR = "ai-forever/ruBert-large-best" #"mbert-token-cls-best" #/checkpoint-3054"
CKPT_DIR = f"{MODEL_NAME}-best"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR, use_fast=True)
model = AutoModelForTokenClassification.from_pretrained(CKPT_DIR).to(device).eval()

# robust id2label (ключи могут быть str/int)
id2label = model.config.id2label
if isinstance(next(iter(id2label.keys())), str):
    id2label = {int(k): v for k, v in id2label.items()}

def predict_spans(context: str, answer: str, max_length: int = 512):
    """
    Возвращает список спанов по 'answer':
    [{'start': int, 'end': int, 'label': str}, ...]
    Индексы start/end даны в символах относительно 'answer'.
    """
    enc = tokenizer(
        context, answer,
        return_offsets_mapping=True,
        truncation=True, max_length=max_length,
        return_tensors="pt"
    )
    seq_ids = enc.sequence_ids(0)                          # 0=context, 1=answer, None=spec
    offsets = enc["offset_mapping"][0].tolist()
    inputs = {k: v.to(device) for k, v in enc.items() if k != "offset_mapping"}

    with torch.no_grad():
        logits = model(**inputs).logits[0]                 # [seq_len, num_labels]
    pred_ids = logits.argmax(-1).tolist()

    spans = []
    cur_label, cur_start, cur_end = None, None, None

    for sid, (st, en), pid in zip(seq_ids, offsets, pred_ids):
        # берём только токены второй последовательности (answer) и не-пустые оффсеты
        if sid != 1 or st == en:
            # если меняется sequence (выходим из answer), закрываем открытый спан
            continue

        lbl = id2label[pid]
        if lbl == "O":
            if cur_label is not None:
                spans.append({"start": cur_start, "end": cur_end, "label": cur_label})
                cur_label, cur_start, cur_end = None, None, None
        else:
            if cur_label is None:
                cur_label, cur_start, cur_end = lbl, st, en
            elif lbl == cur_label:
                cur_end = en
            else:
                spans.append({"start": cur_start, "end": cur_end, "label": cur_label})
                cur_label, cur_start, cur_end = lbl, st, en

    if cur_label is not None:
        spans.append({"start": cur_start, "end": cur_end, "label": cur_label})

    return spans

In [27]:
from copy import deepcopy

def postprocess_spans(spans, threshold: int = 5):
    """
    Сглаживает короткие спаны: любые отрезки длиной <= threshold
    приклеивает к ближайшему разумному соседу (см. правила выше).
    На выходе возвращает непересекающиеся спаны, отсортированные по start,
    с максимально возможным склеиванием одинаковых label.
    """
    if not spans:
        return []
    spans = sorted(deepcopy(spans), key=lambda s: (s["start"], s["end"]))
    n = len(spans)
    res = []

    i = 0
    while i < n:
        s = spans[i]
        length = s["end"] - s["start"]

        if length <= threshold:
            # Крайние случаи
            if n == 1:
                # Нечего приклеивать — просто оставим как есть
                # (или можно выбросить — на усмотрение)
                # Здесь оставим как есть.
                if res and res[-1]["label"] == s["label"] and res[-1]["end"] >= s["start"]:
                    res[-1]["end"] = max(res[-1]["end"], s["end"])
                else:
                    res.append(s)
                i += 1
                continue

            if i == 0:
                # Приклеить к правому
                spans[i+1]["start"] = s["start"]
                i += 1
                continue
            if i == n - 1:
                # Приклеить к левому
                if res:
                    res[-1]["end"] = s["end"]
                else:
                    # теоретически не должно случиться, но на всякий
                    res.append({"start": s["start"], "end": s["end"], "label": s["label"]})
                i += 1
                continue

            # Обычный случай: есть левый и правый
            prev = res[-1] if res else None
            nxt = spans[i+1]

            # Если по каким-то причинам предыдущего еще нет (напр., первый был коротким и приклеился вправо),
            # создадим искусственный prev из spans[i-1] с учетом возможных правок
            if prev is None:
                prev = deepcopy(spans[i-1])
                res.append(prev)

            if prev["label"] == nxt["label"]:
                # Склеиваем prev и nxt, «перепрыгивая» через короткий s
                prev["end"] = max(prev["end"], nxt["end"])
                # пропускаем nxt
                i += 2
                continue
            else:
                # Приклеиваем к более длинному соседу
                left_len = prev["end"] - prev["start"]
                right_len = nxt["end"] - nxt["start"]
                if left_len >= right_len:
                    prev["end"] = max(prev["end"], s["end"])
                    i += 1
                    continue
                else:
                    nxt["start"] = min(nxt["start"], s["start"])
                    i += 1
                    continue

        # Нормальный (не короткий) спан — просто добавляем с возможным схлопыванием
        if res and res[-1]["label"] == s["label"] and res[-1]["end"] >= s["start"]:
            # слить смежные/перекрывающиеся одинаковые лейблы
            res[-1]["end"] = max(res[-1]["end"], s["end"])
        else:
            res.append(deepcopy(s))
        i += 1

    # Финальный проход: схлопнуть возможные соседние одинаковые лейблы
    final = []
    for s in res:
        if final and final[-1]["label"] == s["label"] and final[-1]["end"] >= s["start"]:
            final[-1]["end"] = max(final[-1]["end"], s["end"])
        else:
            final.append(s)
    return final



LabelItem = Dict[str, Any]
ResultDict = Dict[str, Any]

def _norm_label(lbl: str) -> str:
    m = {
        "paraphrase": "equivalent",
        "equivalent": "equivalent",
        "contradiction": "contradiction",
        "addition": "addition",
    }
    return m.get(lbl.strip().lower(), f"__unknown__:{lbl}")

## Match Bidirectional Spans order

In [52]:
def match_bidirectional_spans(
    result_1: ResultDict,
    result_2: ResultDict,
    pair_labels: Tuple[str, ...] = ("equivalent", "contradiction"),
    addition_label: str = "addition",
    return_warnings: bool = True,
) -> Tuple[List[Dict[str, str]], List[str]]:
    """
    Сопоставляет спаны из двух однонаправленных разметок (result_1 и result_2).

    Ожидается, что:
      - result_1["context"] = текст_1, result_1["labels"] = спаны в координатах текста_1
      - result_2["context"] = текст_2, result_2["labels"] = спаны в координатах текста_2
    Функция возвращает список словарей вида:
      {"span_1": <подстрока из текста_1>, "span_2": <подстрока из текста_2 или пусто>, "label": <label>}
    и список предупреждений.

    Логика:
      - 'paraphrase' нормализуем в 'equivalent'.
      - Для label из pair_labels (по умолчанию 'equivalent' и 'contradiction') мачтим спаны по порядку.
      - Для 'addition' создаем записи с пустым вторым спаном в зависимости от стороны.
      - Любые лишние/неизвестные спаны вызывают предупреждения и не попадают в итог (кроме одиночных 'addition').
    """
    warnings: List[str] = []

    text_1 = result_1.get("answer", "")
    text_2 = result_2.get("answer", "")
    spans_1_raw: List[LabelItem] = result_1.get("labels", [])
    spans_2_raw: List[LabelItem] = result_2.get("labels", [])

    # Нормализуем и сортируем по start
    def prep(spans: List[LabelItem], side: str) -> List[LabelItem]:
        prepped = []
        for s in spans:
            lbl = _norm_label(s["label"])
            if lbl.startswith("__unknown__"):
                warnings.append(f"[{side}] неизвестный лейбл '{s['label']}' → пропущен.")
                continue
            prepped.append({"start": s["start"], "end": s["end"], "label": lbl})
        prepped.sort(key=lambda x: (x["start"], x["end"]))
        return prepped

    spans_1 = prep(spans_1_raw, "result_1")
    spans_2 = prep(spans_2_raw, "result_2")

    # Группируем по лейблам
    by_label_1: Dict[str, List[Tuple[int, int]]] = defaultdict(list)
    by_label_2: Dict[str, List[Tuple[int, int]]] = defaultdict(list)

    for s in spans_1:
        by_label_1[s["label"]].append((s["start"], s["end"]))
    for s in spans_2:
        by_label_2[s["label"]].append((s["start"], s["end"]))

    aligned: List[Dict[str, str]] = []

    # Парные лейблы: сопоставляем по порядку
    for lbl in pair_labels:
        lst1 = by_label_1.get(lbl, [])
        lst2 = by_label_2.get(lbl, [])
        m = min(len(lst1), len(lst2))

        for i in range(m):
            (a1, b1) = lst1[i]
            (a2, b2) = lst2[i]
            span_1 = text_1[a1:b1]
            span_2 = text_2[a2:b2]
            aligned.append({"span_1": span_2, "span_2": span_1, "label": lbl})

        if len(lst1) != len(lst2):
            warnings.append(
                f"[{lbl}] разное кол-во спанов: result_1={len(lst1)} vs result_2={len(lst2)}. "
                f"Сопоставлено {m}, лишние отброшены."
            )

    # Одиночные 'addition' с каждой стороны
    if addition_label:
        for (a1, b1) in by_label_1.get(addition_label, []):
            aligned.append({ "span_1": "", "span_2": text_1[a1:b1],"label": addition_label})
        for (a2, b2) in by_label_2.get(addition_label, []):
            aligned.append({ "span_1": text_2[a2:b2], "span_2": "", "label": addition_label})

    # Предупреждения о лейблах, которые остались «сиротами» (не парные и не addition)
    all_known = set(pair_labels) | ({addition_label} if addition_label else set())
    left_others = [k for k in by_label_1.keys() if k not in all_known]
    right_others = [k for k in by_label_2.keys() if k not in all_known]
    for k in left_others:
        warnings.append(f"[result_1] лейбл '{k}' не поддержан агрегацией и отброшен.")
    for k in right_others:
        warnings.append(f"[result_2] лейбл '{k}' не поддержан агрегацией и отброшен.")

    return aligned, warnings

## Match Bidirectional Spans Encoder

In [28]:
from sentence_transformers import SentenceTransformer
import numpy as np
from scipy.optimize import linear_sum_assignment

span_encoder = SentenceTransformer("intfloat/multilingual-e5-large")

def _encode_spans(spans, text):
    # spans: list[(start, end)] в координатах text
    strings = [text[a:b] for (a, b) in spans]
    if not strings:
        return np.zeros((0, 768)), []  # .get_sentence_embedding_dimension()
    emb = span_encoder.encode(strings, normalize_embeddings=True, convert_to_numpy=True)
    return emb, strings

def _align_by_encoder(
    lst1, lst2,
    text_1: str,
    text_2: str,
    label: str,
    sim_threshold: float = 0.8,
):
    """
    lst1, lst2: списки (start, end) для одного label
      - lst1: спаны в text_1 (original)
      - lst2: спаны в text_2 (paraphrased)
    Возвращает aligned-списки вида:
      {"span_1": span_from_text_2, "span_2": span_from_text_1, "label": label}
    """
    E1, spans_1_str = _encode_spans(lst1, text_1)  # original
    E2, spans_2_str = _encode_spans(lst2, text_2)  # paraphrased

    if E1.shape[0] == 0 or E2.shape[0] == 0:
        return [], 0  # alignments, matched_count=0

    # косинус, если эмбеддинги нормализованы
    S = E1 @ E2.T  # [n1, n2]
    cost = -S
    row_ind, col_ind = linear_sum_assignment(cost)

    aligned_local = []
    matched_pairs = 0
    for i, j in zip(row_ind, col_ind):
        if S[i, j] < sim_threshold:
            continue
        # Сохраняем семантику: span_1 из text_2, span_2 из text_1
        span_from_text_1 = spans_1_str[i]
        span_from_text_2 = spans_2_str[j]
        aligned_local.append({
            "span_1": span_from_text_2,
            "span_2": span_from_text_1,
            "label": label,
        })
        matched_pairs += 1

    return aligned_local, matched_pairs

In [29]:
def match_bidirectional_spans(
    result_1: ResultDict,
    result_2: ResultDict,
    pair_labels: Tuple[str, ...] = ("equivalent", "contradiction"),
    addition_label: str = "addition",
    return_warnings: bool = True,
) -> Tuple[List[Dict[str, str]], List[str]]:
    """
    Сопоставляет спаны из двух однонаправленных разметок (result_1 и result_2).

    Ожидается, что:
      - result_1["context"] = текст_1, result_1["labels"] = спаны в координатах текста_1
      - result_2["context"] = текст_2, result_2["labels"] = спаны в координатах текста_2

    Возвращает:
      aligned: список словарей вида
        {"span_1": <подстрока из текста_2>, "span_2": <подстрока из текста_1 или пусто>, "label": <label>}
      warnings: список предупреждений (если return_warnings=True)

    Логика:
      - 'paraphrase' нормализуем в 'equivalent'.
      - Для label из pair_labels (по умолчанию 'equivalent' и 'contradiction')
        мачтим спаны с помощью энкодера (mE5) через assignment по косинусной близости.
      - Для 'addition' создаем записи с пустым вторым спаном в зависимости от стороны.
      - Любые лишние/неизвестные спаны вызывают предупреждения и не попадают в итог (кроме одиночных 'addition').
    """
    warnings: List[str] = []

    text_1 = result_1.get("answer", "")
    text_2 = result_2.get("answer", "")
    spans_1_raw: List[LabelItem] = result_1.get("labels", [])
    spans_2_raw: List[LabelItem] = result_2.get("labels", [])

    # Нормализуем и сортируем по start
    def prep(spans: List[LabelItem], side: str) -> List[LabelItem]:
        prepped = []
        for s in spans:
            lbl = _norm_label(s["label"])
            if lbl.startswith("__unknown__"):
                if return_warnings:
                    warnings.append(f"[{side}] неизвестный лейбл '{s['label']}' → пропущен.")
                continue
            prepped.append({"start": s["start"], "end": s["end"], "label": lbl})
        prepped.sort(key=lambda x: (x["start"], x["end"]))
        return prepped

    spans_1 = prep(spans_1_raw, "result_1")
    spans_2 = prep(spans_2_raw, "result_2")

    # Группируем по лейблам
    by_label_1: Dict[str, List[Tuple[int, int]]] = defaultdict(list)
    by_label_2: Dict[str, List[Tuple[int, int]]] = defaultdict(list)

    for s in spans_1:
        by_label_1[s["label"]].append((s["start"], s["end"]))
    for s in spans_2:
        by_label_2[s["label"]].append((s["start"], s["end"]))

    aligned: List[Dict[str, str]] = []

    # Парные лейблы: сопоставляем через encoder
    for lbl in pair_labels:
        lst1 = by_label_1.get(lbl, [])
        lst2 = by_label_2.get(lbl, [])

        if not lst1 or not lst2:
            # если с одной стороны нет спанов такого типа — просто warning
            if return_warnings and (lst1 or lst2):
                warnings.append(
                    f"[{lbl}] спаны есть только с одной стороны: "
                    f"result_1={len(lst1)} vs result_2={len(lst2)}. "
                    f"Все такие спаны отброшены."
                )
            continue

        aligned_local, matched_pairs = _align_by_encoder(
            lst1, lst2, text_1, text_2, lbl, sim_threshold=0.8
        )
        aligned.extend(aligned_local)

        if return_warnings and matched_pairs < max(len(lst1), len(lst2)):
            warnings.append(
                f"[{lbl}] сопоставлено {matched_pairs} из "
                f"result_1={len(lst1)}, result_2={len(lst2)} спанов "
                f"(по порогу similarity). Несопоставленные отброшены."
            )

    # Одиночные 'addition' с каждой стороны
    if addition_label:
        for (a1, b1) in by_label_1.get(addition_label, []):
            # addition в text_1 → спан в original → кладем в span_2
            aligned.append({
                "span_1": "",
                "span_2": text_1[a1:b1],
                "label": addition_label,
            })
        for (a2, b2) in by_label_2.get(addition_label, []):
            # addition в text_2 → спан в paraphrased → кладем в span_1
            aligned.append({
                "span_1": text_2[a2:b2],
                "span_2": "",
                "label": addition_label,
            })

    # Предупреждения о лейблах, которые остались (не pair_labels и не addition)
    all_known = set(pair_labels) | ({addition_label} if addition_label else set())
    left_others = [k for k in by_label_1.keys() if k not in all_known]
    right_others = [k for k in by_label_2.keys() if k not in all_known]
    if return_warnings:
        for k in left_others:
            warnings.append(f"[result_1] лейбл '{k}' не поддержан агрегацией и отброшен.")
        for k in right_others:
            warnings.append(f"[result_2] лейбл '{k}' не поддержан агрегацией и отброшен.")

    return aligned, warnings

## Predict

In [30]:
from tqdm import tqdm
import pandas as pd

df = pd.read_csv("../dataset/rulesm_paraphrased.csv")
df = df.dropna()
preds = []
THRESHOLD = 10

for i, row in tqdm(df.iterrows(), total=len(df)):
    # if i != 7:
    #     continue
    try:
        inp_1 = row['paraphrased_paragraph_1']
        inp_2 = row['paragraph_2']
        
        labels_1 = predict_spans(inp_1, inp_2)
        labels_1_processed = postprocess_spans(labels_1, threshold=THRESHOLD)
        result_1 = {"context": inp_1, "answer": inp_2, "labels": labels_1_processed}
        
        labels_2 = predict_spans(inp_2, inp_1)
        labels_2_processed = postprocess_spans(labels_2, threshold=THRESHOLD)
        result_2 = {"context": inp_2, "answer": inp_1, "labels": labels_2_processed}
    
        gold_like = match_bidirectional_spans(result_1, result_2)[0]
        preds.append(gold_like)
    except ValueError as E:
        print(E)
        preds.append(None)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 49/49 [00:04<00:00, 10.66it/s]


In [31]:
df['output'] = preds

In [55]:
preds[0] # при Weighted CE

[{'span_1': '4) по иску государственного или местного органа власти, которому закон предоставляет право требовать ликвидации юридического лица,',
  'span_2': '4) по иску государственного органа или органа местного самоуправления, которым право на предъявление требования о ликвидации юридического лица предоставлено законом,',
  'label': 'equivalent'},
 {'span_1': ' общественным движением, благотворительным и иным фон',
  'span_2': ' общественным',
  'label': 'equivalent'},
 {'span_1': ' в случае систематического осуществления общественной организацией,',
  'span_2': ' в случае систематического осуществления общественной организацией,',
  'label': 'contradiction'},
 {'span_1': 'дом, религиозной организацией деятельности, противоречащей уставным целям таких организаций;',
  'span_2': ' движением,',
  'label': 'contradiction'}]

In [75]:
preds[0] # при CE

[{'span_1': '4) по иску государственного или местного органа власти, которому закон предоставляет право требовать ликвидации юридического лица,',
  'span_2': '4) по иску государственного органа или органа местного самоуправления, которым право на предъявление требования о ликвидации юридического лица предоставлено законом,',
  'label': 'equivalent'},
 {'span_1': ' в случае систематического осуществления общественной организацией, общественным движением, благотворительным и иным фондом, религиозной организацией деятельности, противоречащей уставным целям таких организаций;',
  'span_2': ' в случае систематического осуществления общественной организацией, общественным движением, общественно полезным фондом, религиозной организацией деятельности, противоречащей уставным целям таких организаций;',
  'label': 'contradiction'}]

In [17]:
preds[0] # rubert

[{'span_1': 'которому закон предоставляет право требовать ликвидации юридического лица,',
  'span_2': 'которым право на предъявление требования о ликвидации юридического лица предоставлено законом,',
  'label': 'equivalent'},
 {'span_1': '4) по иску государственного или местного органа власти,',
  'span_2': '4) по иску государственного органа или органа местного самоуправления,',
  'label': 'contradiction'},
 {'span_1': 'в случае систематического осуществления общественной организацией, общественным движением, благотворительным и иным фондом, религиозной организацией деятельности, противоречащей уставным целям таких организаций;',
  'span_2': 'в случае систематического осуществления общественной организацией, общественным движением, общественно полезным фондом, религиозной организацией деятельности, противоречащей уставным целям таких организаций;',
  'label': 'contradiction'}]

In [32]:
preds[0] # e5

[{'span_1': '4) по иску государственного или местного органа власти, которому закон предоставляет право требовать ликвидации юридического лица,',
  'span_2': '4) по иску государственного органа или органа местного самоуправления, которым право на предъявление требования о ликвидации юридического лица предоставлено законом,',
  'label': 'equivalent'},
 {'span_1': 'в случае систематического осуществления общественной организацией, общественным движением, благотворительным и иным фондом, религиозной организацией деятельности, противоречащей уставным целям таких организаций;',
  'span_2': 'полезным фондом, религиозной организацией деятельности, противоречащей уставным целям таких организаций;',
  'label': 'contradiction'}]

In [18]:
df.head(3)

,url,paragraph_1,paragraph_2,subset,span_1,span_2,Label,Anchor_span,total,paraphrased,paraphrased_paragraph_1,paraphrased_output,output
0,https://www.consultant.ru/cons/cgi/online.cgi?...,4) по иску государственного органа или органа ...,4) по иску государственного органа или органа ...,Гражданский кодекс,['4) по иску государственного органа или орган...,['4) по иску государственного органа или орган...,"['Equivalent', 'Contradiction']",[''],[{'span_1': '4) по иску государственного орган...,"[""4) по иску государственного или местного орг...",4) по иску государственного или местного орган...,[{'span_1': '4) по иску государственного или м...,[{'span_1': 'которому закон предоставляет прав...
1,https://www.consultant.ru/cons/cgi/online.cgi?...,"Имущество, передаваемое личному фонду его учре...",Имущество личного фонда принадлежит личному фо...,Гражданский кодекс,"['Имущество, передаваемое личному фонду его уч...",['Имущество личного фонда принадлежит личному ...,"['Contradiction', 'Equivalent', 'Addition']","['', 'Имущество, передаваемое личному фонду ег...","[{'span_1': 'Имущество, передаваемое личному ф...","[""Имущество, передаваемое личному фонду его уч...","Имущество, передаваемое личному фонду его учре...","[{'span_1': 'Имущество, передаваемое личному ф...","[{'span_1': 'Имущество, передаваемое личному ф..."
2,https://www.consultant.ru/cons/cgi/online.cgi?...,2. Налоговый орган обязан осуществить постанов...,2. Налоговый орган обязан осуществить постанов...,Налоговый кодекс,['2. Налоговый орган обязан осуществить постан...,['2. Налоговый орган обязан осуществить постан...,"['Equivalent', 'Contradiction', 'Addition', 'E...","['', '', 'и в тот же срок выдать ему свидетель...",[{'span_1': '2. Налоговый орган обязан осущест...,"[""2. Налоговая служба обязана поставить физиче...",2. Налоговая служба обязана поставить физическ...,[{'span_1': '2. Налоговая служба обязана поста...,[{'span_1': '2. Налоговая служба обязана поста...


In [33]:
df.to_csv('mmbert_sft_on_rules_paraphrased_e5large_me5.csv', index=False)

In [34]:
labels_1

[{'start': 0, 'end': 832, 'label': 'paraphrase'},
 {'start': 832, 'end': 919, 'label': 'contradiction'},
 {'start': 919, 'end': 947, 'label': 'paraphrase'},
 {'start': 947, 'end': 949, 'label': 'contradiction'},
 {'start': 949, 'end': 994, 'label': 'paraphrase'},
 {'start': 994, 'end': 1026, 'label': 'contradiction'},
 {'start': 1026, 'end': 1031, 'label': 'paraphrase'},
 {'start': 1031, 'end': 1040, 'label': 'contradiction'},
 {'start': 1040, 'end': 1136, 'label': 'paraphrase'}]

In [35]:
labels_2

[{'start': 0, 'end': 701, 'label': 'paraphrase'},
 {'start': 701, 'end': 710, 'label': 'contradiction'},
 {'start': 710, 'end': 717, 'label': 'paraphrase'},
 {'start': 717, 'end': 726, 'label': 'contradiction'},
 {'start': 726, 'end': 729, 'label': 'paraphrase'},
 {'start': 729, 'end': 742, 'label': 'contradiction'},
 {'start': 742, 'end': 748, 'label': 'paraphrase'},
 {'start': 748, 'end': 755, 'label': 'contradiction'},
 {'start': 755, 'end': 758, 'label': 'paraphrase'},
 {'start': 758, 'end': 768, 'label': 'contradiction'},
 {'start': 768, 'end': 859, 'label': 'paraphrase'},
 {'start': 859, 'end': 886, 'label': 'contradiction'},
 {'start': 886, 'end': 888, 'label': 'addition'},
 {'start': 888, 'end': 903, 'label': 'contradiction'},
 {'start': 903, 'end': 908, 'label': 'addition'},
 {'start': 908, 'end': 911, 'label': 'contradiction'},
 {'start': 911, 'end': 912, 'label': 'addition'},
 {'start': 912, 'end': 975, 'label': 'contradiction'},
 {'start': 975, 'end': 976, 'label': 'additio

In [22]:
df.iloc[2]['output']

[{'span_1': '2. Налоговая служба обязана поставить физическое лицо на учет, исходя из его заявления, поданного согласно пунктам 6, 7 или 7.2 статьи 83 данного Кодекса, в течение пяти дней с даты приема заявления налоговым органом и в тот же срок выдать ему свидетельство о постановке на учет в налоговом органе (если ранее указанное свидетельство не выдавалось) или уведомление о постановке на учет В случае, если заявление физического лица отправлено по почте заказным письмом или электронными средствами связи через канал связи или портал государственных услуг, налоговая служба осуществляет учет физического лица по такому заявлению в течение пяти дней с момента получения подтверждения данных из заявления от органов, указанных в пунктах 3 и 8 статьи 85 этого Кодекса и в тот же срок выдает (направляет) физическому лицу свидетельство о постановке на учет в налоговом органе (если ранее указанное свидетельство не выдавалось) или уведомление о постановке на учет',
  'span_2': '2. Налоговый орган

In [26]:
df.iloc[2]['output']

[{'span_1': '2. Налоговая служба обязана поставить физическое лицо на учет, исходя из его заявления, поданного согласно пунктам 6, 7 или 7.2 статьи 83 данного Кодекса, в течение пяти дней с даты приема заявления налоговым органом и в тот же срок выдать ему свидетельство о постановке на учет в налоговом органе (если ранее указанное свидетельство не выдавалось) или уведомление о постановке на учет В случае, если заявление физического лица отправлено по почте заказным письмом или электронными средствами связи через канал связи или портал государственных услуг, налоговая служба осуществляет учет физического лица по такому заявлению в течение пяти дней с момента получения подтверждения данных из заявления от органов, указанных в пунктах 3 и 8 статьи 85 этого Кодекса и в тот же срок выдает (направляет) физическому лицу свидетельство о постановке на учет в налоговом органе (если ранее указанное свидетельство не выдавалось) или уведомление о постановке на учет',
  'span_2': '2. Налоговый орган

In [23]:
row = df.iloc[2]
THRESHOLD = 0
inp_1 = row['paraphrased_paragraph_1']
inp_2 = row['paragraph_2']

labels_1 = predict_spans(inp_1, inp_2)
labels_1_processed = postprocess_spans(labels_1, threshold=THRESHOLD)
result_1 = {"context": inp_1, "answer": inp_2, "labels": labels_1_processed}

labels_2 = predict_spans(inp_2, inp_1)
labels_2_processed = postprocess_spans(labels_2, threshold=THRESHOLD)
result_2 = {"context": inp_2, "answer": inp_1, "labels": labels_2_processed}

In [28]:
result_1

{'context': '2. Налоговая служба обязана поставить физическое лицо на учет, исходя из его заявления, поданного согласно пунктам 6, 7 или 7.2 статьи 83 данного Кодекса, в течение пяти дней с даты приема заявления налоговым органом и в тот же срок выдать ему свидетельство о постановке на учет в налоговом органе (если ранее указанное свидетельство не выдавалось) или уведомление о постановке на учет В случае, если заявление физического лица отправлено по почте заказным письмом или электронными средствами связи через канал связи или портал государственных услуг, налоговая служба осуществляет учет физического лица по такому заявлению в течение пяти дней с момента получения подтверждения данных из заявления от органов, указанных в пунктах 3 и 8 статьи 85 этого Кодекса и в тот же срок выдает (направляет) физическому лицу свидетельство о постановке на учет в налоговом органе (если ранее указанное свидетельство не выдавалось) или уведомление о постановке на учет',
 'answer': '2. Налоговый орган 

In [29]:
result_2

{'context': '2. Налоговый орган обязан осуществить постановку на учет физического лица на основании заявления этого физического лица, поданного в соответствии с пунктами 6, 7 или 7.2 статьи 83 настоящего Кодекса, в течение пяти дней со дня получения указанного заявления налоговым органом и в тот же срок выдать (направить) ему выписку из Единого государственного реестра налогоплательщиков, содержащую сведения о постановке на учет в налоговом органе. В случае, если заявление физического лица направлено по почте заказным письмом либо передано в электронной форме по телекоммуникационным каналам связи или с использованием единого портала государственных и муниципальных услуг в налоговый орган, налоговый орган осуществляет постановку на учет физического лица на основании такого заявления в течение пяти дней со дня получения от органов, указанных в пунктах 3 и 8 статьи 85 настоящего Кодекса, подтверждения содержащихся в этом заявлении сведений и в тот же срок выдает (направляет) физическому л

In [30]:
result_2['answer'][794:863]

' физическому лицу свидетельство о постановке на учет в налоговом орга'

In [31]:
eval(row['paraphrased_output'])

[{'span_1': '2. Налоговая служба обязана поставить физическое лицо на учет, исходя из его заявления, поданного согласно пунктам 6, 7 или 7.2 статьи 83 данного Кодекса, в течение пяти дней с даты приема заявления налоговым органом',
  'span_2': '2. Налоговый орган обязан осуществить постановку на учет физического лица на основании заявления этого физического лица, поданного в соответствии с пунктами 6, 7 или 7.2 статьи 83 настоящего Кодекса, в течение пяти дней со дня получения указанного заявления налоговым органом',
  'label': 'Equivalent'},
 {'span_1': 'и в тот же срок выдать ему свидетельство о постановке на учет в налоговом органе',
  'span_2': 'и в тот же срок выдать (направить) ему выписку из Единого государственного реестра налогоплательщиков, содержащую сведения о постановке на учет в налоговом органе.',
  'label': 'Contradiction'},
 {'span_1': '(если ранее указанное свидетельство не выдавалось) или уведомление о постановке на учет',
  'span_2': '',
  'label': 'Addition',
  'an

In [24]:
eval(row['paraphrased_output'])

[{'span_1': '2. Налоговая служба обязана поставить физическое лицо на учет, исходя из его заявления, поданного согласно пунктам 6, 7 или 7.2 статьи 83 данного Кодекса, в течение пяти дней с даты приема заявления налоговым органом',
  'span_2': '2. Налоговый орган обязан осуществить постановку на учет физического лица на основании заявления этого физического лица, поданного в соответствии с пунктами 6, 7 или 7.2 статьи 83 настоящего Кодекса, в течение пяти дней со дня получения указанного заявления налоговым органом',
  'label': 'Equivalent'},
 {'span_1': 'и в тот же срок выдать ему свидетельство о постановке на учет в налоговом органе',
  'span_2': 'и в тот же срок выдать (направить) ему выписку из Единого государственного реестра налогоплательщиков, содержащую сведения о постановке на учет в налоговом органе.',
  'label': 'Contradiction'},
 {'span_1': '(если ранее указанное свидетельство не выдавалось) или уведомление о постановке на учет',
  'span_2': '',
  'label': 'Addition',
  'an